In [ ]:
!pip install sklearn-som

In [ ]:
import pandas as pd
import numpy as np
import time
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & Evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Supervised Algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier # ANN

# Unsupervised Algorithms (If you choose to benchmark them)
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture # EM
from sklearn_som.som import SOM

In [ ]:
# Load + Preprocessing
# Map the labels and fix errors

def map_label(label):
    if label == 'BENIGN':
        return 0
    if 'Brute Force' in label:
        return 1
    if 'XSS' in label:
        return 2
    if 'Sql Injection' in label:
        return 3
    return np.nan

def load_and_preprocess_data():

  # Load Dataset
  df = pd.read_csv('/content/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')

  # Clean column names and drop broken rows
  df.columns = df.columns.str.strip()
  df.replace([np.inf, -np.inf], np.nan, inplace=True)
  df.dropna(inplace=True)
  print(df['Label'].unique())


  print("Completed Data Loading!")


  df['Label'] = df['Label'].map(map_label)
  df_filtered = df[df['Label'].notna()].copy()
  df_filtered['Label'] = df_filtered['Label'].astype(int)

  assert df_filtered['Label'].nunique() > 1, "Label filtering matched only one class"
  print(df_filtered['Label'].value_counts())

  # Extract target variable (Answer Key)
  y = df_filtered['Label'].values
  print(np.unique(y, return_counts=True))
  X_raw = df_filtered.drop(columns=['Label'])

  # Drop non-numeric features
  X_numeric = X_raw.select_dtypes(include=[np.number])

  print("Normalizing features")
  scaler = StandardScaler()
  X_scaled = scaler.fit_transform(X_numeric)

  print("Clipping outliers to [-3, 3]")
  X_final = np.clip(X_scaled, -3, 3)

  print(f"Preprocessing complete. Final Data Shape: {X_final.shape}")
  return X_final, y

X, y = load_and_preprocess_data()

['BENIGN' 'Web Attack � Brute Force' 'Web Attack � XSS'
 'Web Attack � Sql Injection']
Completed Data Loading!
Label
0    168051
1      1507
2       652
3        21
Name: count, dtype: int64
(array([0, 1, 2, 3]), array([168051,   1507,    652,     21]))
Normalizing features
Clipping outliers to [-3, 3]
Preprocessing complete. Final Data Shape: (170231, 78)


In [ ]:
# Models + Tuning

# Put models in dictionary with settings listed in paper
models = {

    # === Supervised ===

    # ANN
    "ANN": MLPClassifier(max_iter=500, random_state=42),

    # Decision Tree -- Best model: criterion = empty, max depth = none, class weight = balanced
    "Decision Tree": DecisionTreeClassifier(criterion='entropy', max_depth=None, class_weight='balanced', random_state=42),

    # k-NN -- 5 models 1-5 k neighbors
    "k-NN (k=1)": KNeighborsClassifier(n_neighbors=1),
    "k-NN (k=2)": KNeighborsClassifier(n_neighbors=2),
    "k-NN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "k-NN (k=4)": KNeighborsClassifier(n_neighbors=4),
    "k-NN (k=5)": KNeighborsClassifier(n_neighbors=5),

    # Naive Bayes -- Default
    "Naive Bayes": GaussianNB(),

    # Random Forest --  Best model: n esimators = 100, max depth = none, class weight = balanced
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=None, class_weight='balanced', random_state=42),

    # SVM -- Best model: kernel = RBF, max iter = -1 (No limit), class weight = balanced
    # Limit to 1000 iterations
    "SVM": SVC(kernel='rbf', class_weight='balanced', max_iter=1000, random_state=42),

    # === Unsupervised ===

    # K-Means -- Default
    "K-Means": KMeans(n_clusters=4, random_state=42),

    # EM -- Default
    "EM": GaussianMixture(n_components=4, random_state=42),

    # SOM -- Default
    "SOM": SOM(m=2, n=2, dim=X.shape[1])
}

In [ ]:
# Evaluation

def evaluate_models(models, X, y):
  results = []

  # 5-Fold cross-validation
  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  for name, model in models.items():
    print(f"Evaluating {name}")

    metrics = {'acc': [], 'prec': [], 'rec': [], 'f1': [], 't1': [], 't2': []}

    for train_idx, test_idx in skf.split(X,y):
      X_train, X_test = X[train_idx], X[test_idx]
      y_train, y_test = y[train_idx], y[test_idx]

      # Measure Training Time (T1)
      start_train = time.perf_counter()

      # Don't feed y_train to unsupervised models
      if name in ["K-Means", "EM", "SOM"]:
        model.fit(X_train)
      else:
        model.fit(X_train, y_train)

      end_train = time.perf_counter()

      # Measure Testing Time (T2)
      start_test = time.perf_counter()
      y_pred = model.predict(X_test)
      end_test = time.perf_counter()

      # Calculate metrics using weighted avg
      metrics['acc'].append(accuracy_score(y_test, y_pred))
      metrics['prec'].append(precision_score(y_test, y_pred, average='weighted', zero_division=0))
      metrics['rec'].append(recall_score(y_test, y_pred, average='weighted', zero_division=0))
      metrics['f1'].append(f1_score(y_test, y_pred, average='weighted', zero_division=0))
      metrics['t1'].append(end_train - start_train)
      metrics['t2'].append(end_test - start_test)

    results.append({
        "Algorithm": name,
        "Accuracy": np.mean(metrics['acc']),
        "Precision": np.mean(metrics['prec']),
        "Recall": np.mean(metrics['rec']),
        "F1-Score": np.mean(metrics['f1']),
        "T1(s)": np.mean(metrics['t1']),
        "T2(s)": np.mean(metrics['t2'])
    })

  return pd.DataFrame(results)

print("\nRunning Benchmarks\n")

benchmark_results = evaluate_models(models, X, y)

benchmark_results = benchmark_results.round({
    'Accuracy': 4, 'Precision': 4, 'Recall': 4, 'F1-Score': 4,
    'T1(s)': 4, 'T2(s)': 4
})

print("Overall Performance of the ML-AIDS Algorithms ===")
display(benchmark_results)




Running Benchmarks

Evaluating ANN
Evaluating Decision Tree
Evaluating k-NN (k=1)
Evaluating k-NN (k=2)
Evaluating k-NN (k=3)
Evaluating k-NN (k=4)
Evaluating k-NN (k=5)
Evaluating Naive Bayes
Evaluating Random Forest
Evaluating SVM
Evaluating K-Means
Evaluating EM
Evaluating SOM
Overall Performance of the ML-AIDS Algorithms ===


,Algorithm,Accuracy,Precision,Recall,F1-Score,T1(s),T2(s)
0,ANN,0.9925,0.9939,0.9925,0.9909,83.5984,0.1220
1,Decision Tree,0.9956,0.9957,0.9956,0.9957,2.8120,0.0067
2,k-NN (k=1),0.9952,0.9952,0.9952,0.9952,0.0222,44.3212
3,k-NN (k=2),0.9957,0.9950,0.9957,0.9950,0.0213,44.5334
4,k-NN (k=3),0.9954,0.9951,0.9954,0.9952,0.0212,44.6190
5,k-NN (k=4),0.9957,0.9950,0.9957,0.9950,0.0256,44.0870
6,k-NN (k=5),0.9954,0.9950,0.9954,0.9951,0.0238,44.0449
7,Naive Bayes,0.9883,0.9902,0.9883,0.9885,0.2174,0.0679
8,Random Forest,0.9956,0.9952,0.9956,0.9953,20.3281,0.1426
9,SVM,0.7670,0.9874,0.7670,0.8588,39.1379,10.9934
